# BraTS VLM Fine-tuning — IT → SFT + MARC

**Design decision:** This is a new notebook rather than an extension of
`brats_inference_colab_new_fixed.ipynb` because the pipeline changes are
architectural, not incremental:
- Two-stage training (IT → SFT) instead of a single QLoRA pass
- Structured JSON output instead of free-text reports
- MARC candidate reranking at inference time
- Different metric stack (structured F1 primary, real GREEN via flan-t5-base)
- Different ground truth source (`btreport_brats23.json` GPT-o3 reports)

**Before running:**
1. Upload `Brats/` folder to `MyDrive/Brats/` (include `brats_data/` and `btreport_brats23.json`)
2. Set Runtime → Change runtime type → **T4 GPU**
3. Run all cells in order

**Estimated runtime on free T4:** ~50–70 min total (IT ~20 min, SFT ~25 min, eval ~10 min)

## 1. Mount Drive and set paths

In [1]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BRATS = '/content/drive/MyDrive/Brats'
WORK_DIR    = '/content/Brats'
import os; os.makedirs(WORK_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Install dependencies (one cell — avoids Colab restart loops)

In [ ]:
# Install all dependencies quietly except qwen-vl-utils (separated so failures are visible)
!pip install -q \
    'transformers>=4.51.0' \
    'peft>=0.14.0' \
    'bitsandbytes>=0.46.1' \
    'accelerate>=0.30.0' \
    'trl>=0.9.0' \
    'datasets>=2.19.0' \
    'GREEN-score' \
    'rouge_score' \
    'bert_score' \
    'nibabel' \
    'Pillow' \
    'einops'
# qwen-vl-utils installed separately — errors would be hidden by -q above
!pip install qwen-vl-utils
print('All dependencies installed.')

# Colab: newly installed packages require a runtime restart to become importable.
# This block restarts once automatically; on the second run packages are already
# loaded so the restart is skipped.
import sys
if 'rouge_score' not in sys.modules:
    print('Restarting runtime to activate new packages — re-run all cells from the top.')
    import os; os.kill(os.getpid(), 9)


ERROR: Ignored the following versions that require a different python version: 0.0.12 Requires-Python ==3.12.1
ERROR: Could not find a version that satisfies the requirement GREEN-score (from versions: none)
ERROR: No matching distribution found for GREEN-score


## 3. Check GPU

In [2]:
import torch
assert torch.cuda.is_available(), 'No GPU found — set Runtime > T4 GPU first.'
print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

GPU  : Tesla T4
VRAM : 15.6 GB


## 4. Select smallest available Qwen3-VL model (or fall back to Qwen2.5-VL-3B)

In [3]:
import re
from huggingface_hub import list_models

def get_smallest_qwen3_vl():
    try:
        models = list(list_models(author='Qwen', search='Qwen3-VL-Instruct'))
    except Exception as e:
        print(f'HF query failed ({e}). Falling back to Qwen2.5-VL-3B-Instruct.')
        return 'Qwen/Qwen2.5-VL-3B-Instruct'
    candidates = []
    for m in models:
        match = re.search(r'Qwen3-VL-([\d.]+)B', m.modelId, re.IGNORECASE)
        if match:
            candidates.append((float(match.group(1)), m.modelId))
    if not candidates:
        print('WARNING: No Qwen3-VL models found. Falling back to Qwen2.5-VL-3B-Instruct.')
        return 'Qwen/Qwen2.5-VL-3B-Instruct'
    candidates.sort(key=lambda x: x[0])
    chosen_size, chosen_id = candidates[0]
    print(f'Selected : {chosen_id}  ({chosen_size}B params)')
    print(f'Available: {[c[1] for c in candidates]}')
    if chosen_size > 4:
        print(f'WARNING: {chosen_size}B may OOM at 4-bit on T4. '
              f'Confirm before proceeding or fall back to Qwen2.5-VL-3B.')
    return chosen_id

# Qwen3-VL loads without error under Qwen2_5_VLForConditionalGeneration but has
# different window-attention parameters → shape crash at runtime. Use 2.5-VL-3B
# which is architecturally compatible and already cached from the first run.
MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
print(f'MODEL_ID = {MODEL_ID}')

MODEL_ID = Qwen/Qwen2.5-VL-3B-Instruct


## 5. File paths

In [4]:
from pathlib import Path

PATHS = {
    'ground_truth'  : f'{DRIVE_BRATS}/btreport_brats23.json',
    'brats_data_dir': f'{WORK_DIR}/brats_data',
    'patient_ids'   : [
        'BraTS-GLI-00000-000',
        'BraTS-GLI-00002-000',
        'BraTS-GLI-00003-000',
        'BraTS-GLI-00005-000',
        'BraTS-GLI-00006-000',
    ],
    'output_dir'    : f'{DRIVE_BRATS}/checkpoints/',
    'model_id'      : MODEL_ID,
    'untuned_dir'   : f'{DRIVE_BRATS}/reports',
    'finetuned_dir' : f'{DRIVE_BRATS}/reports_finetuned',
}

EVAL_PATIENT   = 'BraTS-GLI-00006-000'
TRAIN_PATIENTS = [p for p in PATHS['patient_ids'] if p != EVAL_PATIENT]
Path(PATHS['output_dir']).mkdir(parents=True, exist_ok=True)
print(f'Train: {TRAIN_PATIENTS}')
print(f'Eval : {EVAL_PATIENT}')

Train: ['BraTS-GLI-00000-000', 'BraTS-GLI-00002-000', 'BraTS-GLI-00003-000', 'BraTS-GLI-00005-000']
Eval : BraTS-GLI-00006-000


## 6. Error isolation — `safe_run` and `FAILURE_LOG`

In [5]:
import traceback
from typing import Any, Callable

FAILURE_LOG: list[dict] = []

def safe_run(fn: Callable, *args, label: str = '', fallback: Any = None, **kwargs) -> Any:
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        tb = traceback.format_exc()
        entry = {'label': label or fn.__name__, 'exception': type(e).__name__,
                 'message': str(e), 'traceback': tb}
        FAILURE_LOG.append(entry)
        print(f'  WARNING [{entry["label"]}] {entry["exception"]}: {entry["message"]}')
        return fallback

def print_failure_summary():
    print('\n' + '='*60)
    print(f'FAILURE SUMMARY — {len(FAILURE_LOG)} issue(s) recorded')
    print('='*60)
    if not FAILURE_LOG:
        print('  No failures. All operations completed successfully.')
    else:
        for i, e in enumerate(FAILURE_LOG, 1):
            print(f'\n[{i}] {e["label"]}')
            print(f'    {e["exception"]}: {e["message"]}')
            if e['exception'] not in ('ImportError', 'ModuleNotFoundError'):
                for line in e['traceback'].strip().split('\n')[-3:]:
                    print(f'    {line}')
    print('='*60 + '\n')

print('safe_run and FAILURE_LOG ready.')

safe_run and FAILURE_LOG ready.


## 7. Check scorer availability

In [6]:
SCORER_AVAILABLE = {}

def check_scorer(import_fn, name):
    result = safe_run(import_fn, label=f'Import: {name}', fallback=None)
    SCORER_AVAILABLE[name] = result is not None
    print(f'  {name}: {"OK" if SCORER_AVAILABLE[name] else "UNAVAILABLE — will skip"}')
    return result

# GREEN uses google/flan-t5-base — small, fits on T4 alongside inference
def _import_green():
    from green_score import GREEN
    return GREEN(model_id_or_path='google/flan-t5-base', do_sample=False)

def _import_rouge():
    from rouge_score import rouge_scorer as rs
    return rs.RougeScorer(['rougeL'], use_stemmer=True)

def _import_bertscore():
    from bert_score import score
    return score

# radgraph requires PhysioNet gated access — expected to be UNAVAILABLE
def _import_radgraph():
    from radgraph import F1RadGraph
    return F1RadGraph(reward_level='partial')

print('Checking scorer availability:')
GREEN_SCORER    = check_scorer(_import_green,    'GREEN')
ROUGE_SCORER    = check_scorer(_import_rouge,    'ROUGE-L')
BERTSCORE_FN    = check_scorer(_import_bertscore,'BERTScore')
RADGRAPH_SCORER = check_scorer(_import_radgraph, 'RadGraph-F1')

Checking scorer availability:
  WARNING [Import: GREEN] ModuleNotFoundError: No module named 'green_score'
  GREEN: UNAVAILABLE — will skip
  WARNING [Import: ROUGE-L] ModuleNotFoundError: No module named 'rouge_score'
  ROUGE-L: UNAVAILABLE — will skip
  WARNING [Import: BERTScore] ModuleNotFoundError: No module named 'bert_score'
  BERTScore: UNAVAILABLE — will skip
  WARNING [Import: RadGraph-F1] ModuleNotFoundError: No module named 'radgraph'
  RadGraph-F1: UNAVAILABLE — will skip


## 8. Copy `brats_data` from Drive to local storage (faster NIfTI I/O)

In [7]:
import shutil

src = Path(DRIVE_BRATS) / 'brats_data'
dst = Path(WORK_DIR) / 'brats_data'
if dst.exists():
    print('brats_data already in local storage — skipping copy.')
else:
    print('Copying brats_data from Drive...')
    shutil.copytree(str(src), str(dst))
    print('Done.')

patient_dirs = sorted(
    d for d in dst.iterdir()
    if d.is_dir() and d.name.startswith('BraTS-')
)
print(f'Patients found: {[d.name for d in patient_dirs]}')

brats_data already in local storage — skipping copy.
Patients found: ['BraTS-GLI-00000-000', 'BraTS-GLI-00002-000', 'BraTS-GLI-00003-000', 'BraTS-GLI-00005-000', 'BraTS-GLI-00006-000']


## 9. Load ground truth and parse structured fields

`btreport_brats23.json` contains GPT-o3-120B and Llama3-70B generated reports.
We use the GPT-o3 reports as the higher-quality reference and extract structured
fields via regex for IT QA generation and SFT JSON targets.

In [8]:
import json, re

with open(PATHS['ground_truth']) as f:
    gt_data = json.load(f)

GT_KEY = 'Predicted Report (gpt-oss:120b)'
print(f'GT entries: {len(gt_data)}   Using key: "{GT_KEY}"')

def parse_gt(report_text: str) -> dict:
    """Extract structured fields from a GPT-o3 radiology report via regex."""
    t = report_text.lower()

    # Midline shift magnitude
    ms = re.search(r'(\d+(?:\.\d+)?)\s*mm[^.]*?midline shift', t)
    if not ms:
        ms = re.search(r'midline shift[^.]*?(\d+(?:\.\d+)?)\s*mm', t)
    midline_mm = float(ms.group(1)) if ms else None

    # Shift direction
    if 'left-to-right' in t or 'leftward' in t:
        shift_dir = 'left-to-right'
    elif 'right-to-left' in t or 'rightward' in t:
        shift_dir = 'right-to-left'
    else:
        shift_dir = None

    # Tumor dimensions
    dims = re.search(r'(\d+\.?\d*)\s*[xX]\s*(\d+\.?\d*)\s*[xX]\s*(\d+\.?\d*)\s*cm', t)
    tumor_dims = [float(d) for d in dims.groups()] if dims else None

    # Necrosis %
    nec = re.search(r'(\d+)\s*%[^.]*?necros|necros[^.]*?(\d+)\s*%', t)
    necrosis_pct = float((nec.group(1) or nec.group(2))) if nec else None

    # Edema %
    edm = re.search(r'(\d+)\s*%[^.]*?edema|edema[^.]*?(\d+)\s*%', t)
    edema_pct = float((edm.group(1) or edm.group(2))) if edm else None

    # Enhancing %
    enh = re.search(r'(\d+)\s*%[^.]*?enhanc|enhanc[^.]*?(\d+)\s*%', t)
    enhancing_pct = float((enh.group(1) or enh.group(2))) if enh else None

    # Boolean findings
    ependymal  = 'ependymal invasion' in t and 'no ependymal' not in t
    vent_eff   = ('effacement' in t or 'effaced' in t) and 'no focal effacement' not in t
    edema_mid  = 'crosses the midline' in t or 'across the midline' in t or 'bilateral' in t
    sulcal_eff = 'sulcal effacement' in t or 'sulci are effaced' in t

    # Deep structures
    deep_structs = [s for s in
        ['caudate','putamen','pallidum','thalamus','insula','accumbens','internal capsule']
        if s in t]

    # Lateralization
    if any(x in t for x in ['left frontal','left temporal','left hemisphere','left parietal']):
        lat = 'left'
    elif any(x in t for x in ['right frontal','right temporal','right hemisphere']):
        lat = 'right'
    else:
        lat = 'unknown'

    # Diagnosis
    if 'glioblastoma' in t or ' gbm' in t:
        diag = 'GBM'
    else:
        diag = 'High-grade glioma'

    return {
        'midline_shift_mm'     : midline_mm,
        'shift_direction'      : shift_dir,
        'ventricular_effacement': vent_eff,
        'ependymal_invasion'   : ependymal,
        'tumor_dimensions_cm'  : tumor_dims,
        'necrosis_pct'         : necrosis_pct,
        'edema_pct'            : edema_pct,
        'enhancing_pct'        : enhancing_pct,
        'edema_crosses_midline': edema_mid,
        'deep_structures'      : deep_structs,
        'lateralization'       : lat,
        'sulcal_effacement'    : sulcal_eff,
        'diagnosis'            : diag,
        'report_text'          : report_text,
    }

# Parse all patients and verify
parsed_gt = {}
for pid in PATHS['patient_ids']:
    if pid not in gt_data:
        print(f'WARNING: {pid} not in ground truth JSON')
        continue
    parsed_gt[pid] = parse_gt(gt_data[pid][GT_KEY])

# Print parsed fields for the eval patient as a sanity check
g = parsed_gt[EVAL_PATIENT]
print(f'\nParsed GT for {EVAL_PATIENT}:')
for k, v in g.items():
    if k != 'report_text':
        print(f'  {k:<28} = {v}')

GT entries: 1251   Using key: "Predicted Report (gpt-oss:120b)"

Parsed GT for BraTS-GLI-00006-000:
  midline_shift_mm             = 8.0
  shift_direction              = left-to-right
  ventricular_effacement       = True
  ependymal_invasion           = True
  tumor_dimensions_cm          = [4.3, 4.3, 6.0]
  necrosis_pct                 = 19.0
  edema_pct                    = None
  enhancing_pct                = 19.0
  edema_crosses_midline        = True
  deep_structures              = ['caudate', 'putamen', 'accumbens']
  lateralization               = left
  sulcal_effacement            = False
  diagnosis                    = High-grade glioma


## 10. Video packing pipeline

Adds a sagittal midline reference frame as frame 0 to give the model an explicit
anchor for left-right hemisphere comparison — directly addressing the midline
shift detection failure in the baseline runs.

In [9]:
import numpy as np
import nibabel as nib
from PIL import Image
import warnings

MODALITIES      = ('t1c', 't1n', 't2f', 't2w')
SLICE_SIZE      = (112, 112)   # must be a multiple of 14 for Qwen2.5-VL patch embedding
TRAIN_MAX_SLICES = 12   # 12 × 4 = 48 frames during training (saves VRAM)
INFER_MAX_SLICES = 48   # 48 × 4 = 192 frames during inference
FRAME_MIN_PIXELS = 112 * 112   # 12544 = (8×14)², valid for Qwen2.5-VL patch embedding
FRAME_MAX_PIXELS = 112 * 112

def find_nifti(patient_dir: Path, modality: str):
    for pat in [f'*-{modality}.nii', f'*-{modality}.nii.gz']:
        matches = list(patient_dir.glob(pat))
        if matches:
            return str(matches[0])
    return None

def vol_to_frames(nii_path: str, max_slices: int) -> list:
    nii  = nib.load(nii_path)
    data = nii.get_fdata(dtype=np.float32)
    zooms = np.array(nii.header.get_zooms()[:3])
    ax   = int(np.argmax(zooms))
    data = np.moveaxis(data, ax, 0)
    mask = data.reshape(data.shape[0], -1).any(axis=1)
    data = data[mask]
    if data.shape[0] == 0:
        return []
    n = min(data.shape[0], max_slices)
    idx = np.round(np.linspace(0, data.shape[0]-1, n)).astype(int)
    data = data[idx]
    vmin, vmax = data.min(), data.max()
    data = ((data - vmin) / (vmax - vmin + 1e-6) * 255).astype(np.uint8)
    return [Image.fromarray(s, mode='L').convert('RGB').resize(SLICE_SIZE, Image.LANCZOS)
            for s in data]

def extract_midsagittal(patient_dir: Path) -> Image.Image:
    """Extract T1n sagittal midline slice as orientation anchor (frame 0)."""
    nii_path = find_nifti(patient_dir, 't1n')
    if not nii_path:
        return Image.new('RGB', SLICE_SIZE, 0)
    nii  = nib.load(nii_path)
    vol  = nii.get_fdata(dtype=np.float32)
    mid  = vol.shape[0] // 2
    slc  = vol[mid, :, :]
    vmin, vmax = slc.min(), slc.max()
    slc  = ((slc - vmin) / (vmax - vmin + 1e-6) * 255).astype(np.uint8)
    return Image.fromarray(slc, mode='L').convert('RGB').resize(SLICE_SIZE, Image.LANCZOS)

def pack_patient_frames(patient_dir: Path, max_slices: int) -> list:
    """[midsagittal_ref] + all 4 modality frames concatenated."""
    frames = [extract_midsagittal(patient_dir)]
    for mod in MODALITIES:
        nii_path = find_nifti(patient_dir, mod)
        if not nii_path:
            warnings.warn(f'{patient_dir.name}: {mod} not found — skipping.')
            continue
        frames.extend(vol_to_frames(nii_path, max_slices))
    return frames

def save_frames(patient_dir: Path, frames: list, suffix: str = '') -> list:
    """Save PIL frames to local disk and return paths for qwen-vl-utils."""
    out_dir = Path(WORK_DIR) / 'frames' / (patient_dir.name + suffix)
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for i, img in enumerate(frames):
        p = out_dir / f'frame_{i:04d}.png'
        if not p.exists():
            img.save(str(p))
        paths.append(str(p))
    return paths

# Pre-extract and cache training frames (48/vol) + inference frames (48/vol)
print('Pre-extracting frames for all patients...')
patient_dir_map = {d.name: d for d in Path(PATHS['brats_data_dir']).iterdir()
                   if d.is_dir() and d.name.startswith('BraTS-')}

train_frame_paths = {}  # pid -> list of PNG paths (48 frames)
infer_frame_paths = {}  # pid -> list of PNG paths (193 frames)

for pid in PATHS['patient_ids']:
    pdir = patient_dir_map.get(pid)
    if not pdir:
        print(f'  WARNING: {pid} directory not found'); continue
    train_frame_paths[pid] = save_frames(pdir, pack_patient_frames(pdir, TRAIN_MAX_SLICES), '_train')
    infer_frame_paths[pid] = save_frames(pdir, pack_patient_frames(pdir, INFER_MAX_SLICES), '_infer')
    print(f'  {pid}: {len(train_frame_paths[pid])} train frames, '
          f'{len(infer_frame_paths[pid])} infer frames')

print('Frame extraction complete.')

Pre-extracting frames for all patients...


/tmp/ipykernel_6778/3048885580.py:49: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  return Image.fromarray(slc, mode='L').convert('RGB').resize(SLICE_SIZE, Image.LANCZOS)
/tmp/ipykernel_6778/3048885580.py:35: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  return [Image.fromarray(s, mode='L').convert('RGB').resize(SLICE_SIZE, Image.LANCZOS)


  BraTS-GLI-00000-000: 49 train frames, 193 infer frames
  BraTS-GLI-00002-000: 49 train frames, 193 infer frames
  BraTS-GLI-00003-000: 49 train frames, 193 infer frames
  BraTS-GLI-00005-000: 49 train frames, 193 infer frames
  BraTS-GLI-00006-000: 49 train frames, 193 infer frames
Frame extraction complete.


## 11. Stage 1 — IT: generate QA pairs from parsed ground truth

Templated QA pairs (no LLM needed). Teaches the model to answer specific
spatial/quantitative questions before the full-report SFT stage.

In [10]:
SYSTEM_IT = ('You are an expert neuroradiologist interpreting multi-sequence '
             'brain MRI studies. Answer questions strictly based on the imaging '
             'provided. Use precise medical terminology. Output only the answer.')

def generate_qa_pairs(pid: str, g: dict) -> list:
    pairs = []
    # Yes/No
    pairs += [
        {'q': 'Is midline shift present in this MRI?',
         'a': 'Yes' if g['midline_shift_mm'] else 'No'},
        {'q': 'Is ependymal invasion present?',
         'a': 'Yes' if g['ependymal_invasion'] else 'No'},
        {'q': 'Does the edema cross the midline?',
         'a': 'Yes' if g['edema_crosses_midline'] else 'No'},
        {'q': 'Is ventricular effacement present?',
         'a': 'Yes' if g['ventricular_effacement'] else 'No'},
        {'q': 'Is sulcal effacement present?',
         'a': 'Yes' if g['sulcal_effacement'] else 'No'},
    ]
    # Factual extraction
    if g['midline_shift_mm']:
        pairs.append({'q': 'What is the magnitude of the midline shift in mm?',
                      'a': str(g['midline_shift_mm'])})
    if g['tumor_dimensions_cm']:
        d = g['tumor_dimensions_cm']
        pairs.append({'q': 'What are the tumor dimensions in centimeters (AP x TV x CC)?',
                      'a': f"{d[0]} x {d[1]} x {d[2]} cm"})
    # Classification
    pairs += [
        {'q': 'Which hemisphere contains the primary tumor?',
         'a': g['lateralization'].capitalize()},
        {'q': 'What is the most likely diagnosis based on imaging?',
         'a': g['diagnosis']},
    ]
    # Quantification
    if g['necrosis_pct'] is not None:
        pairs.append({'q': 'What percentage of the lesion volume represents necrosis?',
                      'a': f"Approximately {g['necrosis_pct']:.0f}%"})
    if g['edema_pct'] is not None:
        pairs.append({'q': 'What percentage of the total abnormality represents vasogenic edema?',
                      'a': f"Approximately {g['edema_pct']:.0f}%"})
    # Contradiction (rejects false claims)
    pairs += [
        {'q': 'True or False: There is no midline shift in this scan.',
         'a': 'False' if g['midline_shift_mm'] else 'True'},
        {'q': 'True or False: The tumor shows no contrast enhancement.',
         'a': 'False'},  # all BraTS GLI cases are enhancing
    ]
    # Sentence completion
    if g['midline_shift_mm'] and g['shift_direction']:
        pairs.append({
            'q': f"Complete: There is a ____ mm {g['shift_direction']} midline shift "
                 f"at the level of the ____.",
            'a': f"{g['midline_shift_mm']:.0f} mm {g['shift_direction']} midline shift "
                 f"at the level of the septum pellucidum"
        })
    return pairs

it_dataset = []
for pid in TRAIN_PATIENTS:
    if pid not in parsed_gt:
        continue
    g = parsed_gt[pid]
    for qa in generate_qa_pairs(pid, g):
        it_dataset.append({'pid': pid, 'frames': train_frame_paths[pid],
                           'question': qa['q'], 'answer': qa['a']})

print(f'IT dataset: {len(it_dataset)} QA pairs across {len(TRAIN_PATIENTS)} patients')
for pid in TRAIN_PATIENTS:
    n = sum(1 for x in it_dataset if x['pid'] == pid)
    print(f'  {pid}: {n} pairs')

IT dataset: 54 QA pairs across 4 patients
  BraTS-GLI-00000-000: 13 pairs
  BraTS-GLI-00002-000: 14 pairs
  BraTS-GLI-00003-000: 13 pairs
  BraTS-GLI-00005-000: 14 pairs


## 12. Stage 1 — IT training (QLoRA r=8, 3 epochs)

In [12]:
!pip install --force-reinstall bitsandbytes>=0.46.1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torchvision 0.25.0+cu128 requires torch==2.10.0, but you have torch 2.11.0 which is incompatible.
libraft-cu12 26.2.0 requires cuda-toolkit[cublas,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
cuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.4 which is incompatible.
torchaudio 2.10.0+cu128 requires torch==2.10.0, but you have torch 2.11.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.3.0 which is incompatible.
cudf-cu12 26.2.1 requires cuda-toolkit[nvcc,nvrtc]==12.*, but you have cuda-toolkit 13.0.2 which is 

In [11]:
import gc
import torch
from transformers import AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from torch.optim import AdamW
from qwen_vl_utils import process_vision_info

# ── Load model in 4-bit ───────────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)

def load_vlm(model_id, bnb_cfg=None):
    # Qwen3-VL and Qwen2.5-VL share Qwen2_5_VLForConditionalGeneration.
    # Always use it directly; fall back to Qwen2.5-VL-3B only if the
    # requested model fails to load (e.g. auth / quota issues).
    kwargs = dict(trust_remote_code=True,
                  quantization_config=bnb_cfg,
                  device_map='auto')
    from transformers import Qwen2_5_VLForConditionalGeneration
    try:
        return Qwen2_5_VLForConditionalGeneration.from_pretrained(model_id, **kwargs)
    except Exception as e:
        print(f'Load of {model_id} failed ({e}), falling back to Qwen2.5-VL-3B.')
        return Qwen2_5_VLForConditionalGeneration.from_pretrained(
            'Qwen/Qwen2.5-VL-3B-Instruct', **kwargs)

print(f'Loading {MODEL_ID} in 4-bit for IT stage...')
model_it = load_vlm(MODEL_ID, bnb_config)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

model_it = prepare_model_for_kbit_training(model_it, use_gradient_checkpointing=True)
for name, param in model_it.named_parameters():
    if 'visual' in name:
        param.requires_grad = False

lora_it = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05, bias='none',
    task_type=TaskType.CAUSAL_LM,
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
)
model_it = get_peft_model(model_it, lora_it)
model_it.print_trainable_parameters()

# ── Label masking helper ──────────────────────────────────────────────────
def make_labels(input_ids, proc):
    header_ids = proc.tokenizer.encode('<|im_start|>assistant\n',
                                       add_special_tokens=False)
    ids = input_ids.tolist()
    labels = [-100] * len(ids)
    n = len(header_ids)
    for i in range(len(ids) - n, -1, -1):
        if ids[i:i+n] == header_ids:
            labels[i+n:] = ids[i+n:]
            break
    return torch.tensor(labels, device=input_ids.device)

# ── IT training loop ──────────────────────────────────────────────────────
IT_EPOCHS  = 3
IT_LR      = 2e-4
GRAD_ACCUM = 8  # effective batch = 8 samples

optimizer = AdamW([p for p in model_it.parameters() if p.requires_grad],
                  lr=IT_LR, weight_decay=0.01)
model_it.train()
optimizer.zero_grad()

for epoch in range(IT_EPOCHS):
    sep = '='*56
    print(f'\n{sep}\n  IT Epoch {epoch+1}/{IT_EPOCHS}\n{sep}')
    epoch_loss, step = 0.0, 0

    for i, sample in enumerate(it_dataset):
        messages = [
            {'role': 'system', 'content': SYSTEM_IT},
            {'role': 'user', 'content': [
                {'type': 'video', 'video': sample['frames'], 'min_pixels': FRAME_MIN_PIXELS, 'max_pixels': FRAME_MAX_PIXELS,
                 'nframes': len(sample['frames'])},
                {'type': 'text',
                 'text': f'Frame 0 is the T1n sagittal midline reference.\n'
                         f'Modalities: T1c (2-{TRAIN_MAX_SLICES+1}), '
                         f'T1n ({TRAIN_MAX_SLICES+2}-{2*TRAIN_MAX_SLICES+1}), '
                         f'T2-FLAIR ({2*TRAIN_MAX_SLICES+2}-{3*TRAIN_MAX_SLICES+1}), '
                         f'T2w ({3*TRAIN_MAX_SLICES+2}-{4*TRAIN_MAX_SLICES+1}).\n\n'
                         f'Question: {sample["question"]}'},
            ]},
            {'role': 'assistant', 'content': sample['answer']},
        ]

        text_in = processor.apply_chat_template(messages, tokenize=False,
                                                 add_generation_prompt=False)
        img_in, vid_in = process_vision_info(messages)
        inputs = processor(text=[text_in], images=img_in, videos=vid_in,
                           padding=True, return_tensors='pt').to('cuda')
        labels = make_labels(inputs['input_ids'][0], processor).unsqueeze(0)

        outputs = model_it(**inputs, labels=labels)
        loss = outputs.loss / GRAD_ACCUM
        loss.backward()
        epoch_loss += loss.item() * GRAD_ACCUM

        del inputs, outputs, labels; torch.cuda.empty_cache()
        step += 1

        if step % GRAD_ACCUM == 0 or (i+1) == len(it_dataset):
            torch.nn.utils.clip_grad_norm_(
                [p for p in model_it.parameters() if p.requires_grad], 1.0)
            optimizer.step(); optimizer.zero_grad()

        if (i+1) % 8 == 0 or (i+1) == len(it_dataset):
            print(f'  step {i+1}/{len(it_dataset)}  '
                  f'loss={loss.item()*GRAD_ACCUM:.4f}')

    print(f'  Epoch {epoch+1} mean loss: {epoch_loss/len(it_dataset):.4f}')

print('\nIT training complete.')

Loading Qwen/Qwen2.5-VL-3B-Instruct in 4-bit for IT stage...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Load of Qwen/Qwen2.5-VL-3B-Instruct failed (Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`), falling back to Qwen2.5-VL-3B.


ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

## 13. Save IT checkpoint to Drive + failure summary

In [ ]:
IT_CKPT = PATHS['output_dir'] + 'it_checkpoint/'
Path(IT_CKPT).mkdir(parents=True, exist_ok=True)
model_it.save_pretrained(IT_CKPT)
processor.save_pretrained(IT_CKPT)
print(f'IT checkpoint saved → {IT_CKPT}')
print_failure_summary()

## 14. Stage 2 — SFT: build structured JSON targets

Each training sample: all frames → structured JSON report.
Loss weighting: 2× on spatial finding tokens, 0.5× on `modality_gaps` tokens.

In [ ]:
SYSTEM_SFT = ('You are an expert neuroradiologist. Given multi-sequence brain MRI '
              'frames, generate a structured JSON radiology report. Include all '
              'fields even if a finding is absent (use false/null). Route any '
              'findings about absent modalities (SWI, DWI) to the modality_gaps '
              'array. Do not fabricate measurements. Respond with valid JSON only.')

SPATIAL_TOKENS = [
    'midline_shift_present','shift_mm','effacement_present',
    'ependymal_invasion','dimensions_cm','necrosis_pct',
    'edema_pct','crosses_midline','deep_structures',
    'entrapment_concern','sulcal_effacement',
]

def build_sft_target(g: dict) -> dict:
    """Convert parsed GT fields into the structured JSON schema."""
    return {
        'mass_effect': {
            'midline_shift_present': g['midline_shift_mm'] is not None,
            'shift_mm'             : g['midline_shift_mm'],
            'shift_direction'      : g['shift_direction'],
            'reference_level'      : 'septum pellucidum',
        },
        'ventricles': {
            'effacement_present': g['ventricular_effacement'],
            'effaced_side'      : g['lateralization'] if g['ventricular_effacement'] else None,
        },
        'tumor': {
            'location'            : f"{g['lateralization']} frontal lobe",
            'lateralization'      : g['lateralization'],
            'dimensions_cm'       : g['tumor_dimensions_cm'],
            'enhancement_pattern' : 'ring',
            'necrosis_present'    : g['necrosis_pct'] is not None and g['necrosis_pct'] > 0,
            'necrosis_pct'        : g['necrosis_pct'],
            'enhancing_pct'       : g['enhancing_pct'],
            'ependymal_invasion'  : g['ependymal_invasion'],
            'deep_structures'     : g['deep_structures'],
        },
        'edema': {
            'present'        : g['edema_pct'] is not None and g['edema_pct'] > 0,
            'pct'            : g['edema_pct'],
            'crosses_midline': g['edema_crosses_midline'],
        },
        'sulcal_effacement': g['sulcal_effacement'],
        'herniation'       : False,
        'diagnosis'        : g['diagnosis'],
        'modality_gaps'    : [
            'SWI not provided — susceptibility/hemorrhage not assessable',
            'DWI not provided — restricted diffusion not assessable',
        ],
    }

def make_weighted_labels(input_ids, proc, spatial_tokens, boost=2.0, gap_weight=0.5):
    """Labels with 2x weight on spatial finding tokens, 0.5x on modality_gaps."""
    base_labels = make_labels(input_ids, proc)
    weights = torch.ones_like(base_labels, dtype=torch.float)
    ids = input_ids.tolist()
    # Find spatial token positions and boost their weight in the labels
    for token_str in spatial_tokens:
        tok_ids = proc.tokenizer.encode(token_str, add_special_tokens=False)
        n = len(tok_ids)
        for i in range(len(ids) - n):
            if ids[i:i+n] == tok_ids and base_labels[i] != -100:
                weights[i:i+n] = boost
    gap_ids = proc.tokenizer.encode('modality_gaps', add_special_tokens=False)
    n = len(gap_ids)
    in_gap = False
    for i, tok in enumerate(ids):
        if ids[i:i+n] == gap_ids:
            in_gap = True
        if in_gap and base_labels[i] != -100:
            weights[i] = gap_weight
    return base_labels, weights

sft_dataset = []
for pid in TRAIN_PATIENTS:
    if pid not in parsed_gt:
        continue
    sft_dataset.append({
        'pid'   : pid,
        'frames': train_frame_paths[pid],
        'target': json.dumps(build_sft_target(parsed_gt[pid]), indent=2),
    })

print(f'SFT dataset: {len(sft_dataset)} samples')
print('\nSample SFT target:')
print(sft_dataset[0]['target'][:600])

## 15. Stage 2 — SFT training (load IT → merge → QLoRA r=16, 5 epochs)

In [ ]:
from peft import PeftModel

# ── Free IT model and reload base fresh for merge ─────────────────────────
del model_it; gc.collect(); torch.cuda.empty_cache()
print(f'VRAM before merge: {torch.cuda.memory_allocated()/1e9:.1f} GB')

print('Loading base model in bfloat16 for adapter merge...')
base_for_merge = load_vlm(MODEL_ID, None)  # no 4-bit for merge
# Cast to bfloat16 explicitly
base_for_merge = base_for_merge.to(torch.bfloat16)

# Attach IT adapter and merge into base weights
model_merged = PeftModel.from_pretrained(base_for_merge, IT_CKPT)
model_merged = model_merged.merge_and_unload()  # returns plain nn.Module
print(f'VRAM after merge: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# Now quantize the merged model to 4-bit for SFT
# Save merged weights to a temp dir, then reload in 4-bit
MERGED_TMP = PATHS['output_dir'] + 'merged_tmp/'
Path(MERGED_TMP).mkdir(parents=True, exist_ok=True)
model_merged.save_pretrained(MERGED_TMP)
processor.save_pretrained(MERGED_TMP)
del model_merged; gc.collect(); torch.cuda.empty_cache()
print('Merged model saved. Reloading in 4-bit for SFT...')

model_sft = load_vlm(MERGED_TMP, bnb_config)
model_sft = prepare_model_for_kbit_training(model_sft, use_gradient_checkpointing=True)
for name, param in model_sft.named_parameters():
    if 'visual' in name:
        param.requires_grad = False

lora_sft = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
    task_type=TaskType.CAUSAL_LM,
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
)
model_sft = get_peft_model(model_sft, lora_sft)
model_sft.print_trainable_parameters()
print(f'VRAM after SFT model load: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# ── SFT training loop ─────────────────────────────────────────────────────
SFT_EPOCHS = 5
SFT_LR     = 1e-4

optimizer_sft = AdamW([p for p in model_sft.parameters() if p.requires_grad],
                      lr=SFT_LR, weight_decay=0.01)

# Cosine LR schedule
from torch.optim.lr_scheduler import CosineAnnealingLR
total_steps = (SFT_EPOCHS * len(sft_dataset) + GRAD_ACCUM - 1) // GRAD_ACCUM
scheduler   = CosineAnnealingLR(optimizer_sft, T_max=total_steps, eta_min=1e-6)

model_sft.train(); optimizer_sft.zero_grad()

for epoch in range(SFT_EPOCHS):
    sep = '='*56
    print(f'\n{sep}\n  SFT Epoch {epoch+1}/{SFT_EPOCHS}\n{sep}')
    epoch_loss, step = 0.0, 0

    for i, sample in enumerate(sft_dataset):
        messages = [
            {'role': 'system', 'content': SYSTEM_SFT},
            {'role': 'user', 'content': [
                {'type': 'video', 'video': sample['frames'], 'min_pixels': FRAME_MIN_PIXELS, 'max_pixels': FRAME_MAX_PIXELS,
                 'nframes': len(sample['frames'])},
                {'type': 'text',
                 'text': 'Available sequences: T1c, T1n, T2-FLAIR, T2w. '
                         'Frame 0 is the sagittal midline reference. '
                         'Generate a structured JSON radiology report.'},
            ]},
            {'role': 'assistant', 'content': sample['target']},
        ]

        text_in = processor.apply_chat_template(messages, tokenize=False,
                                                 add_generation_prompt=False)
        img_in, vid_in = process_vision_info(messages)
        inputs = processor(text=[text_in], images=img_in, videos=vid_in,
                           padding=True, return_tensors='pt').to('cuda')

        labels, weights = make_weighted_labels(
            inputs['input_ids'][0], processor, SPATIAL_TOKENS)
        labels  = labels.unsqueeze(0)
        weights = weights.unsqueeze(0).to('cuda')

        # Weighted loss: multiply per-token loss by spatial weights
        outputs   = model_sft(**inputs, labels=labels)
        base_loss = outputs.loss  # mean over unmasked tokens (PyTorch default)
        # Re-compute token-level loss with weights
        logits = outputs.logits[:, :-1].contiguous()
        tgt    = labels[:, 1:].contiguous()
        w      = weights[:, 1:].contiguous()
        token_loss = torch.nn.functional.cross_entropy(
            logits.view(-1, logits.size(-1)), tgt.view(-1),
            ignore_index=-100, reduction='none')
        w_flat = w.view(-1)
        mask   = tgt.view(-1) != -100
        loss   = (token_loss * w_flat * mask).sum() / (mask.sum() + 1e-6)
        loss   = loss / GRAD_ACCUM
        loss.backward()
        epoch_loss += loss.item() * GRAD_ACCUM

        del inputs, outputs, labels, weights, logits, tgt, token_loss
        torch.cuda.empty_cache(); step += 1

        if step % GRAD_ACCUM == 0 or (i+1) == len(sft_dataset):
            torch.nn.utils.clip_grad_norm_(
                [p for p in model_sft.parameters() if p.requires_grad], 1.0)
            optimizer_sft.step(); scheduler.step(); optimizer_sft.zero_grad()

        if (i+1) == len(sft_dataset):
            print(f'  Epoch {epoch+1} loss: {epoch_loss/len(sft_dataset):.4f}  '
                  f'lr: {scheduler.get_last_lr()[0]:.2e}')

print('\nSFT training complete.')

## 16. Save SFT checkpoint to Drive + failure summary

In [ ]:
SFT_CKPT = PATHS['output_dir'] + 'sft_checkpoint/'
Path(SFT_CKPT).mkdir(parents=True, exist_ok=True)
model_sft.save_pretrained(SFT_CKPT)
processor.save_pretrained(SFT_CKPT)
print(f'SFT checkpoint saved → {SFT_CKPT}')
print_failure_summary()

## 17. MARC inference on eval patient

Generates N=4 candidates via top-p sampling, scores each for absent-modality
violations (SWI/DWI families), selects the best, and removes violating content.

In [ ]:
from peft import PeftModel as _PeftModel

# Load SFT model for inference (bfloat16 for quality)
del model_sft; gc.collect(); torch.cuda.empty_cache()

base_inf = load_vlm(MERGED_TMP, None)
base_inf = base_inf.to(torch.bfloat16)
model_inf = _PeftModel.from_pretrained(base_inf, SFT_CKPT)
model_inf.eval()

ABSENT_MODALITY_CUES = {
    'DIFFUSION': {
        'sequences'    : ['DWI','ADC','DTI','diffusion'],
        'clinical_cues': ['restricted diffusion','acute infarction',
                          'cytotoxic edema','diffusion restriction'],
        'family_weight': 1.5,
    },
    'SUSCEPTIBILITY': {
        'sequences'    : ['SWI','T2*','susceptibility'],
        'clinical_cues': ['microbleed','hemosiderin','blooming artifact',
                          'susceptibility artifact','blood products',
                          'intratumoral bleeding','hemorrhage on SWI'],
        'family_weight': 1.2,
    },
}

def score_candidate(report_str):
    score, t = 0.0, report_str.lower()
    for family, cfg in ABSENT_MODALITY_CUES.items():
        for cue in cfg['sequences'] + cfg['clinical_cues']:
            if cue.lower() in t:
                # Free pass if correctly placed in modality_gaps
                gaps_idx = t.find('"modality_gaps"')
                if gaps_idx != -1 and cue.lower() in t[gaps_idx:gaps_idx+500]:
                    continue
                ctx = t[max(0,t.find(cue.lower())-50):t.find(cue.lower())+100]
                pol = 1.0 if any(n in ctx for n in ['not ','no ','absent','false']) else 1.5
                score += cfg['family_weight'] * pol
    return score

def _nullify_violations(obj, cues, in_gaps=False):
    if isinstance(obj, dict):
        return {k: _nullify_violations(v, cues, k=='modality_gaps') for k,v in obj.items()}
    elif isinstance(obj, list):
        return [_nullify_violations(x, cues, in_gaps) for x in obj]
    elif isinstance(obj, str) and not in_gaps:
        for cue in cues:
            if cue in obj.lower():
                return 'not assessable — modality not provided'
    return obj

def remove_violations(report_str):
    all_cues = [c.lower() for cfg in ABSENT_MODALITY_CUES.values()
                for c in cfg['clinical_cues']]
    try:
        return json.dumps(_nullify_violations(json.loads(report_str), all_cues), indent=2)
    except json.JSONDecodeError:
        return report_str

def prepare_inputs(frame_paths, proc):
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'video', 'video': frame_paths, 'min_pixels': FRAME_MIN_PIXELS, 'max_pixels': FRAME_MAX_PIXELS,
             'nframes': len(frame_paths)},
            {'type': 'text',
             'text': 'Available sequences: T1c, T1n, T2-FLAIR, T2w. '
                     'Frame 0 is the sagittal midline reference. '
                     'Generate a structured JSON radiology report.'},
        ],
    }]
    text_in = proc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    img_in, vid_in = process_vision_info(messages)
    return proc(text=[text_in], images=img_in, videos=vid_in,
                padding=True, return_tensors='pt').to('cuda')

def marc_inference(frame_paths, proc, n_candidates=4):
    # Pre-process frames once and cache prompt length — avoids
    # re-decoding 193 video frames N times inside the loop.
    inputs = prepare_inputs(frame_paths, proc)
    prompt_len = inputs['input_ids'].shape[1]
    candidates = []
    for _ in range(n_candidates):
        with torch.inference_mode():
            out = model_inf.generate(
                **inputs,
                do_sample=True, top_p=0.95, temperature=0.8, max_new_tokens=800,
            )
        text = proc.decode(out[0][prompt_len:], skip_special_tokens=True).strip()
        candidates.append(text)
        del out; torch.cuda.empty_cache()
    del inputs; torch.cuda.empty_cache()
    scored = sorted([(score_candidate(c), c) for c in candidates], key=lambda x: x[0])
    best_score, best = scored[0]
    if best_score > 0:
        best = remove_violations(best)
    return best, best_score, candidates

print(f'Running MARC inference on {EVAL_PATIENT}...')
pred_str, viol_score, all_cands = safe_run(
    marc_inference, infer_frame_paths[EVAL_PATIENT], processor,
    label=f'MARC {EVAL_PATIENT}', fallback=('{}', -1.0, [])
)
print(f'Violation score: {viol_score}')
print(f'\nPrediction (first 600 chars):\n{pred_str[:600]}')

## 18. Evaluation — structured F1 + RadGraph + GREEN + secondary metrics

All metric calls wrapped in `safe_run`. Failed scorers return `None` — the
evaluation loop always completes regardless of individual scorer failures.

In [ ]:
def extract_nested(d, key, default=None):
    """Recursively search for a key in a nested dict."""
    if not isinstance(d, dict):
        return default
    if key in d:
        return d[key]
    for v in d.values():
        result = extract_nested(v, key, default)
        if result is not default:
            return result
    return default

def compute_structured_f1(pred_json_str, gt):
    try:
        pred = json.loads(pred_json_str)
    except Exception:
        return {'parse_error': True, 'overall_f1': 0.0}
    results = {}
    binary = {
        'midline_shift_present': gt['midline_shift_mm'] is not None,
        'ependymal_invasion'   : gt['ependymal_invasion'],
        'ventricular_effacement': gt['ventricular_effacement'],
        'edema_crosses_midline': gt['edema_crosses_midline'],
        'sulcal_effacement'    : gt['sulcal_effacement'],
    }
    for field, gt_val in binary.items():
        results[field] = int(bool(extract_nested(pred, field)) == bool(gt_val))
    for field, gt_val in [('shift_mm', gt['midline_shift_mm']),
                           ('necrosis_pct', gt['necrosis_pct']),
                           ('edema_pct', gt['edema_pct'])]:
        if gt_val is None:
            continue
        pred_val = extract_nested(pred, field)
        if pred_val is not None:
            rel_err = abs(float(pred_val)-float(gt_val))/(abs(float(gt_val))+1e-6)
            results[field] = int(rel_err <= 0.20)
        else:
            results[field] = 0
    pred_diag = str(extract_nested(pred, 'diagnosis', '')).upper()
    results['diagnosis'] = int(pred_diag in gt['diagnosis'].upper() or
                               gt['diagnosis'].upper() in pred_diag)
    results['overall_f1'] = sum(v for v in results.values()
                                if isinstance(v, (int,float))) / max(len(results)-1, 1)
    return results

def json_to_prose(report_json):
    """Flatten structured JSON to prose for RadGraph/GREEN scoring."""
    if isinstance(report_json, str):
        report_json = json.loads(report_json)
    r = report_json; lines = []
    ms = r.get('mass_effect', {})
    if ms.get('midline_shift_present'):
        lines.append(f"There is a {ms.get('shift_mm','')} mm "
                     f"{ms.get('shift_direction','')} midline shift "
                     f"at the level of the {ms.get('reference_level','')}. ")
    v = r.get('ventricles', {})
    if v.get('effacement_present'):
        lines.append(f"The {v.get('effaced_side','')} lateral ventricle is effaced. ")
    t = r.get('tumor', {})
    if t:
        dims = t.get('dimensions_cm', [])
        dim_str = ' x '.join(str(d) for d in dims)+' cm' if dims else ''
        lines.append(f"A {t.get('enhancement_pattern','')} enhancing mass "
                     f"in the {t.get('location','')} measures {dim_str}. "
                     f"Necrosis: {t.get('necrosis_pct','')}%. ")
        if t.get('ependymal_invasion'):
            lines.append('Ependymal invasion is present. ')
        if t.get('deep_structures'):
            lines.append(f"Deep structures: {', '.join(t['deep_structures'])}. ")
    e = r.get('edema', {})
    if e.get('present'):
        lines.append(f"Vasogenic edema: {e.get('pct','')}%" +
                     (', crosses midline.' if e.get('crosses_midline') else '.'))
    lines.append(f"Diagnosis: {r.get('diagnosis','')}. ")
    return ' '.join(lines)

# ── Full evaluation ───────────────────────────────────────────────────────
gt_eval = parsed_gt[EVAL_PATIENT]
ref_json = build_sft_target(gt_eval)

pred_json = safe_run(json.loads, pred_str, label=f'JSON parse {EVAL_PATIENT}', fallback={})
pred_prose = safe_run(json_to_prose, pred_json, label='Prose convert pred', fallback='')
ref_prose  = safe_run(json_to_prose, ref_json, label='Prose convert ref',  fallback='')

structured = safe_run(compute_structured_f1, pred_str, gt_eval,
                      label=f'Structured F1 {EVAL_PATIENT}',
                      fallback={'overall_f1': None, 'parse_error': True})

radgraph = None
if SCORER_AVAILABLE.get('RadGraph-F1') and pred_prose and ref_prose:
    radgraph = safe_run(
        lambda: RADGRAPH_SCORER(hyps=[pred_prose], refs=[ref_prose])[2][0],
        label=f'RadGraph-F1 {EVAL_PATIENT}', fallback=None)

green = None
if SCORER_AVAILABLE.get('GREEN') and pred_prose and ref_prose:
    green = safe_run(
        lambda: GREEN_SCORER([ref_prose], [pred_prose])[0],
        label=f'GREEN {EVAL_PATIENT}', fallback=None)

rouge_l = None
if SCORER_AVAILABLE.get('ROUGE-L') and pred_prose and ref_prose:
    rouge_l = safe_run(
        lambda: ROUGE_SCORER.score(ref_prose, pred_prose)['rougeL'].fmeasure,
        label=f'ROUGE-L {EVAL_PATIENT}', fallback=None)

bert_f1 = None
if SCORER_AVAILABLE.get('BERTScore') and pred_prose and ref_prose:
    import bert_score.utils as bsu
    from bert_score.utils import model2layers
    _orig = bsu.sent_encode
    def _safe_encode(tok, s):
        if getattr(tok,'model_max_length',0) > 100_000: tok.model_max_length = 512
        return _orig(tok, s)
    bsu.sent_encode = _safe_encode
    try:
        bert_f1 = safe_run(
            lambda: float(BERTSCORE_FN([pred_prose],[ref_prose],lang='en',verbose=False)[2].mean()),
            label=f'BERTScore {EVAL_PATIENT}', fallback=None)
    finally:
        bsu.sent_encode = _orig

result = {
    'patient_id'      : EVAL_PATIENT,
    'structured_f1'   : structured,
    'radgraph_f1'     : radgraph,
    'green'           : green,
    'rouge_l'         : rouge_l,
    'bert_f1'         : bert_f1,
    'marc_violations' : viol_score,
    'pred_prose'      : pred_prose,
}

print(f'\nResults for {EVAL_PATIENT}:')
sf = result['structured_f1'] or {}
print(f'  Structured F1 (overall): {sf.get("overall_f1")}')
for k, v in sf.items():
    if k not in ('overall_f1','parse_error'):
        print(f'    {k:<30} = {v}')
print(f'  RadGraph-F1              : {result["radgraph_f1"]}')
print(f'  GREEN                    : {result["green"]}')
print(f'  ROUGE-L                  : {result["rouge_l"]}')
print(f'  BERTScore-F1             : {result["bert_f1"]}')
print(f'  MARC violations          : {result["marc_violations"]}')

## 19. Compare IT+SFT+MARC vs baselines

Loads the untuned `(1).txt` and previous QLoRA `(2).txt` reports for the eval
patient and computes the same structured F1 for a direct comparison.

In [ ]:
def eval_text_report(report_path, label):
    """Score a stored text/JSON report file."""
    if not Path(report_path).exists():
        print(f'  {label}: file not found — skipping')
        return None
    raw = Path(report_path).read_text(encoding='utf-8')
    text = '\n'.join(l for l in raw.splitlines() if not l.startswith('#')).strip()
    # Try to parse as JSON; fall back to scoring as prose
    structured = safe_run(compute_structured_f1, text, parsed_gt[EVAL_PATIENT],
                          label=f'Structured F1 {label}',
                          fallback={'overall_f1': None, 'parse_error': True})
    return structured

baseline_untuned  = eval_text_report(
    f'{PATHS["untuned_dir"]}/{EVAL_PATIENT}.txt', 'Untuned (1)')
baseline_finetuned = eval_text_report(
    f'{PATHS["finetuned_dir"]}/{EVAL_PATIENT}.txt', 'Prev QLoRA (2)')

it_sft = result['structured_f1'] or {}

print(f'\n{"Metric":<30} {"Untuned":>10} {"Prev QLoRA":>12} {"IT+SFT+MARC":>13}')
print('='*68)
fields = ['midline_shift_present','ependymal_invasion','ventricular_effacement',
          'edema_crosses_midline','sulcal_effacement','diagnosis','overall_f1']
for f in fields:
    u = (baseline_untuned  or {}).get(f, 'N/A')
    p = (baseline_finetuned or {}).get(f, 'N/A')
    s = it_sft.get(f, 'N/A')
    print(f'  {f:<28} {str(u):>10} {str(p):>12} {str(s):>13}')
print('='*68)

## 20. Save all results to Drive + final failure summary

In [ ]:
import pandas as pd

results_dir = Path(DRIVE_BRATS) / 'finetuning_results'
results_dir.mkdir(exist_ok=True)

# Save evaluation CSV
flat = {'patient_id': result['patient_id'],
        'structured_f1_overall': (result['structured_f1'] or {}).get('overall_f1'),
        'radgraph_f1': result['radgraph_f1'],
        'green': result['green'],
        'rouge_l': result['rouge_l'],
        'bert_f1': result['bert_f1'],
        'marc_violations': result['marc_violations']}
pd.DataFrame([flat]).to_csv(results_dir / 'it_sft_marc_results.csv', index=False)

# Save predicted JSON report
(results_dir / f'{EVAL_PATIENT}_IT_SFT_MARC.json').write_text(pred_str)

print(f'Results saved to {results_dir}')
print_failure_summary()